# CBED Frame Simulation

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from cbed_simulation.frame_builder import FrameParameters
from cbed_simulation.crystal_orientation import OrientedPhase, ExperimentInformation

In [ ]:
phase = OrientedPhase.from_cif("./Si.cif", (1, 1, 0))

In [ ]:
experiment = ExperimentInformation(
    frame_shape=(512, 512),
    radius_px=7,
    transmitted_centre_px=complex(256, 256),
    pattern_scale_factor=120.,  # pixels / Å-1
)

In [ ]:
frame_params = FrameParameters(
    disk_blur_sigma=0.5,
    intensity_from_radius=False,
    textured=False,
    inelastic_scatter_sigma=0.,
    additive_noise_scale=0,
)

In [ ]:
def make_frame(experiment, frame_params, stretch_abc=(1., 1., 1.), max_excitation_error=None):
    sim_peaks = phase.peak_positions(experiment, stretch_abc=stretch_abc, max_excitation_error=max_excitation_error)
    # Play with the intensities to make the transmitted beam more similar to the diffracted beams
    sim_peaks.modify_intensities(power=0.25)
    sim_peaks.modify_000_intensity(multiply=2)
    return phase.synthetic(experiment, sim_peaks, frame_params=frame_params), sim_peaks

In [ ]:
frame_vacuum, _ = make_frame(experiment, frame_params, max_excitation_error=1e-5)  # tiny excitation error means only central spot is retained

In [ ]:
from scipy.ndimage import center_of_mass
vac_com = center_of_mass(frame_vacuum)
vac_com

In [ ]:
if not frame_params.textured:
    assert np.allclose(vac_com, (experiment.transmitted_centre_px.imag, experiment.transmitted_centre_px.real), atol=0.1)

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(5, 5))
ax1.imshow(frame_vacuum, cmap="gray")
ax1.plot(vac_com[1], vac_com[1], 'rx')
ax1.set_title("Vacuum probe");

In [ ]:
frame_unstrained, sim_peaks_unstrained = make_frame(experiment, frame_params)
stretch = (1., 1., 1.015)
frame_strained, sim_peaks_strained = make_frame(experiment, frame_params, stretch_abc=stretch)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(frame_unstrained, cmap="gray")
ax1.set_title("Unstrained")
ax1.set_xlim(0, frame_unstrained.shape[1])
ax1.set_ylim(frame_unstrained.shape[0], 0)
sim_peaks_unstrained.to_pixels(experiment).plot(fig, ax1)
ax2.imshow(frame_strained, cmap="gray")
ax2.set_title("Strained");
ax2.set_xlim(0, frame_unstrained.shape[1])
ax2.set_ylim(frame_unstrained.shape[0], 0)
sim_peaks_strained.to_pixels(experiment).plot(fig, ax2)

## Validate strain values

Compute strain value using template matching to check we are applying the right stretches

In [ ]:
import libertem.api as lt
from libertem_blobfinder.common.correlation import get_peaks
from libertem_blobfinder.common.patterns import UserTemplate
from libertem_blobfinder.udf.correlation import FullFrameCorrelationUDF
from libertem_blobfinder.common.gridmatching import Matcher
from strain_mapping.strain_decomposition import compute_strain_large_def


def to_complex(yx) -> complex:
    return complex(yx[1], yx[0])


def match_template(ctx, frame, template, peaks):
    ds = ctx.load("memory", data=frame[np.newaxis, ...])
    udf = FullFrameCorrelationUDF(
        peaks=peaks.astype(int), match_pattern=template, zero_shift=None, upsample=20,
    )
    corr_result = ctx.run_udf(
        dataset=ds,
        udf=udf
    )
    positions = corr_result["refineds"].data.squeeze(axis=0)
    positions_int = corr_result["centers"].data.squeeze(axis=0)
    correlation_score = corr_result["peak_values"].data.squeeze(axis=0)
    return positions, positions_int, correlation_score

In [ ]:
cx, cy = int(experiment.transmitted_centre_px.real), int(experiment.transmitted_centre_px.imag)
rad = int(experiment.radius_px + max(experiment.radius_px // 3, 3))
template = frame_vacuum[cy - rad: cy + rad, cx - rad: cx + rad]
template = np.clip(template, 0, 20)
template = UserTemplate(template)

In [ ]:
template_com = center_of_mass(template.template)
print(f"Template CoM: {template_com}, Template shape: {template.template.shape}, //2 {np.asarray(template.template.shape) // 2}")

In [ ]:
fig, ax = plt.subplots()
ax.imshow(template.template, cmap='gray')
ax.set_title("Template")

In [ ]:
ctx = lt.Context.make_with("inline")

In [ ]:
peaks_unstrained = get_peaks(frame_unstrained, template, 40)
peaks_strained = get_peaks(frame_strained, template, 40)

In [ ]:
positions_unstrained, positions_unstrained_int, score_unstrained = match_template(ctx, frame_unstrained, template, peaks_unstrained)
positions_strained, positions_strained_int, score_strained = match_template(ctx, frame_strained, template, peaks_strained)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(frame_unstrained, cmap='gray')
ax1.plot(positions_unstrained[:, 1], positions_unstrained[:, 0], 'ro')
ax1.plot(positions_strained[:, 1], positions_strained[:, 0], 'go', alpha=0.3)
ax2.imshow(frame_strained, cmap='gray')
ax2.plot(positions_strained[:, 1], positions_strained[:, 0], 'go')
ax2.plot(positions_unstrained[:, 1], positions_unstrained[:, 0], 'ro', alpha=0.3);

In [ ]:
zero = sim_peaks_unstrained.to_pixels(experiment).spot_position((0, 0, 0))
vec_a = sim_peaks_unstrained.to_pixels(experiment).spot_position((0, 0, 2), centre_zero=True)
vec_b = sim_peaks_unstrained.to_pixels(experiment).spot_position((-1, 1, 1), centre_zero=True)
zero = np.asarray((zero.imag, zero.real))
vec_a = np.asarray((vec_a.imag, vec_a.real))
vec_b = np.asarray((vec_b.imag, vec_b.real))
match_unstrained = Matcher().fastmatch(positions_unstrained_int, zero, vec_a, vec_b, refineds=positions_unstrained, peak_values=score_unstrained)
match_strained = Matcher().fastmatch(positions_strained_int, zero, vec_a, vec_b, refineds=positions_strained, peak_values=score_strained)
match_strained.zero

In [ ]:
strain_res = compute_strain_large_def(
    to_complex(match_strained.a),
    to_complex(match_strained.b),
    to_complex(match_unstrained.a),
    to_complex(match_unstrained.b),
)
strain_res_rot = strain_res.to_vector(to_complex(match_strained.a))
strain_res_rot

In [ ]:
print(f"Computed: {strain_res_rot.e_xx * 100:.2f}%, Applied {(stretch[-1] - 1) * 100:.2f}%")